In [5]:
import pandas as pd
import numpy as np


data = pd.read_csv('data/flats_final.csv')

In [6]:
dataset = data.sample(frac=1).reset_index(drop=True)

In [8]:
dataset

,floor,floors_count,rooms_count,total_meters,price,distance_to_subway,distance_to_center,district_rank
0,2,18,1,37.83,8133450,0.000378,10907.980218,15
1,1,10,4,72.50,10600000,0.001817,8513.552863,9
2,2,17,1,50.50,14150226,0.001237,9364.667145,15
3,3,12,2,51.59,13385888,0.000000,12443.018922,9
4,4,19,2,63.30,13500000,0.000424,6283.947924,4
...,...,...,...,...,...,...,...,...
5015,8,9,4,74.80,14000000,0.000844,9442.794295,14
5016,4,6,4,117.10,19600000,0.000886,4007.280559,2
5017,4,8,3,88.83,39055320,0.002717,2612.434052,1
5018,5,10,1,26.00,9200000,0.000000,4432.529967,2


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [11]:
y = data['price']
X = data.drop(['price'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)



In [18]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)



# print(mean_squared_error(lin_reg.predict(X_test), y_test) ** 0.5)

LinearRegression()

In [115]:
y_pred = lin_reg.predict(X_test)

deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 917
87


In [99]:
dec_tree = DecisionTreeRegressor(max_depth=20)

dec_tree.fit(X_train, y_train)


DecisionTreeRegressor(max_depth=20)

In [119]:
y_pred = dec_tree.predict(X_test)

deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 298
706


In [105]:
rand_for = RandomForestRegressor(max_depth=30)
rand_for.fit(X_train, y_train)

RandomForestRegressor(max_depth=30)

In [118]:
y_pred = rand_for.predict(X_test)

deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 3000000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 274
730


У линейной регрессии есть отрицательные предсказания, у деревьев и рандомного леса нет!!!

In [116]:
print((y_pred < 0).sum())

82


Index(['floor', 'floors_count', 'rooms_count', 'total_meters',
       'distance_to_subway', 'distance_to_center', 'district_rank'],
      dtype='object')

In [146]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd


# конвертируем данные в тензоры PyTorch
X_train_t = torch.FloatTensor(X_train.to_numpy())
X_test_t = torch.FloatTensor(X_test.to_numpy())
y_train_t = torch.FloatTensor(y_train.to_numpy())
y_test_t = torch.FloatTensor(y_test.to_numpy())

# создаем DataLoader'ы для обучения и тестирования
train_data = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_data, batch_size=32)
test_data = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_data, batch_size=1)

# определяем модель

class Net(nn.Module):
    def __init__(self, input_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, 140)  # входной слой
        self.relu1 = nn.ReLU()  # функция активации
        self.fc4 = nn.Linear(140, 1)  # выходной слой

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc4(out)
        return out

model = Net(7)

# определяем функцию потерь и оптимизатор
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# обучаем модель
for epoch in range(1000):
    for inputs, targets in train_loader:
        # обнуляем градиенты
        optimizer.zero_grad()
        # прямой проход
        outputs = model(inputs)
        # вычисляем потери
        loss = criterion(outputs, targets)
        # обратный проход
        loss.backward()
        # обновляем веса
        optimizer.step()

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

# оцениваем модель
model.eval()
with torch.no_grad():
    total_loss = 0
    for inputs, targets in test_loader:
        outputs = model(inputs)
        total_loss += 1 if abs(outputs - targets) <= 500000 else 0
        print(outputs, targets)
    print(f'Average loss: {total_loss}')

Epoch 1, Loss: 222237730799616.0
Epoch 2, Loss: 220437854289920.0
Epoch 3, Loss: 217329069719552.0
Epoch 4, Loss: 213004708741120.0
Epoch 5, Loss: 207621252448256.0
Epoch 6, Loss: 201379725443072.0
Epoch 7, Loss: 194493483581440.0
Epoch 8, Loss: 187152310730752.0
Epoch 9, Loss: 179539766411264.0
Epoch 10, Loss: 171831944282112.0
Epoch 11, Loss: 164194972336128.0
Epoch 12, Loss: 156782764752896.0
Epoch 13, Loss: 149735042187264.0
Epoch 14, Loss: 143175402389504.0
Epoch 15, Loss: 137209902530560.0
Epoch 16, Loss: 131925641527296.0
Epoch 17, Loss: 127389719855104.0
Epoch 18, Loss: 123648266469376.0
Epoch 19, Loss: 120725927100416.0
Epoch 20, Loss: 118625537097728.0
Epoch 21, Loss: 117328456974336.0
Epoch 22, Loss: 116795327381504.0
Epoch 23, Loss: 116967612612608.0
Epoch 24, Loss: 117769873915904.0
Epoch 25, Loss: 119112990720000.0
Epoch 26, Loss: 120898002616320.0
Epoch 27, Loss: 123020890865664.0
Epoch 28, Loss: 125377410236416.0
Epoch 29, Loss: 127868021506048.0
Epoch 30, Loss: 1304024